In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from time import perf_counter as timer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load the text chunks and their corresponding embeddings from the CSV file
text_chunks_and_embeddings_df = pd.read_csv("text_chunk_embeddings.csv")

# Convert embeddings from string representation back to numpy arrays
text_chunks_and_embeddings_df["embedding"] = text_chunks_and_embeddings_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
# Convert our embeddings to a PyTorch tensor
embeddings = torch.tensor(np.stack(text_chunks_and_embeddings_df["embedding"].to_list(),axis=0),dtype=torch.float32).to(device)
# Convert our text chunks to a dictionary for easier access
pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")

text_chunks_and_embeddings_df

In [ ]:
embeddings.shape

In [ ]:
# Create Model

from sentence_transformers import SentenceTransformer, util
embedding_model = SentenceTransformer(model_name_or_path='all-mpnet-base-v2', device=device)

In [ ]:
# Define a query
query = "macronutrients functions"
print(f"Query: {query}")

# embed the query
query_embedding = embedding_model.encode(query, convert_to_tensor=True, device=device)

# Get Similarity Scores with dot product
start_time = timer()
dot_scores = util.dot_score(a= query_embedding, b=embeddings)[0]
end_time = timer()
print(f"Time taken for dot product similarity search: {end_time - start_time:.4f} seconds")

# Get top 5 results
top_k = 5
top_k_indices = torch.topk(dot_scores, k=top_k)
top_k_indices

In [ ]:
pages_and_chunks[42]

In [ ]:
import textwrap 

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, width=wrap_length)
    print(wrapped_text)

In [ ]:
print(f"Query: {query}\n")
print("Results: ")

for i, idx in zip(top_k_indices[0], top_k_indices[1]):
    print(f"Score: {i:.4f}")
    print("Text: ")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("-"*80)

In [ ]:
def dot_product(v1, v2):
    return torch.dot(v1, v2)

def cosine_similarity(v1, v2):
    dot_product = torch.dot(v1, v2)

    norm_v1 = torch.sqrt(torch.sum(v1 ** 2))
    norm_v2 = torch.sqrt(torch.sum(v2 ** 2))

    return dot_product / (norm_v1 * norm_v2)

# Example usage
v1 = torch.tensor([1.0, 2.0, 3.0])
v2 = torch.tensor([4.0, 5.0, 6.0])
v3 = torch.tensor([1.0, 2.0, 3.0])
v4 = torch.tensor([-1.0, -2.0, -3.0])
print("Dot Product(v1, v2):", dot_product(v1, v2))
print("Dot Product (v1, v3):", dot_product(v1, v3))
print("Dot Product (v1, v4):", dot_product(v1, v4))
print("Cosine Similarity(v1, v2):", cosine_similarity(v1, v2))
print("Cosine Similarity (v1, v3):", cosine_similarity(v1, v3))
print("Cosine Similarity (v1, v4):", cosine_similarity(v1, v4))

In [ ]:
def retrieve_relevant_resources(query: str,
                                embeddings: torch.Tensor,
                                model: SentenceTransformer= embedding_model,
                                n_resources: int = 5,
                                print_time: bool = True):
    # Embed the query
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Get Similarity Scores with dot product
    start_time = timer()
    dot_scores = util.dot_score(a= query_embedding, b=embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"Time taken for dot product similarity search: {end_time - start_time:.4f} seconds")
    
    # Get top n_resources results
    scores, indices = torch.topk(input=dot_scores, k=n_resources)

    return scores, indices

def print_top_results(query: str,
                    embeddings: torch.Tensor,
                    pages_and_chunks: list[dict] = pages_and_chunks,
                    n_resourcees_to_return: int = 5):
    # 
    scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings, n_resources=n_resourcees_to_return)

    # 
    for i, idx in zip(scores, indices):
        print(f"Score: {i:.4f}")
        print("Text: ")
        print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
        print(f"Page number: {pages_and_chunks[idx]['page_number']}")
        print("-"*80)

In [ ]:
query="foods high in fiber"
# retrieve_relevant_resources(query=query, embeddings=embeddings)
print_top_results(query=query, embeddings=embeddings)

In [ ]:
!nvidia-smi

In [ ]:
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2**30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto"
)

In [ ]:
llm_model

In [ ]:
def get_model_num_parameters(model):
    return sum(p.numel() for p in model.parameters())
num_parameters = get_model_num_parameters(llm_model)
print(f"Number of parameters in the model: {num_parameters:,}")

In [ ]:
input_text = "What are the functions of macronutrients?"
# Create a dialogue template with the input text
dialogue_template = [{
    "role": "user",
    "content": input_text
}]
# Apply the chat template to create the prompt for the model
prompt = tokenizer.apply_chat_template(conversation=dialogue_template, add_generation_prompt=True, tokenizer=False)
print(prompt)

In [ ]:
# Generate a response from the model
# Tokenize the prompt and move it to the appropriate device
input_ids = torch.tensor(prompt["input_ids"], dtype=torch.long, device=device).unsqueeze(0)
attention_mask = torch.tensor(prompt["attention_mask"], dtype=torch.long, device=device).unsqueeze(0)

# Generate a response from the model
output = llm_model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=512)
print(f"Model ouput (token): \n{output}\n")


In [ ]:
# Decode the output tokens to get the generated text
outputs_decoded = tokenizer.decode(output[0])
print(f"Model output (decoded): \n{outputs_decoded}\n")

In [ ]:
# Nutrition-style questions generated with GPT4
gpt4_questions = [
    "What are the macronutrients, and what roles do they play in the human body?",
    "How do vitamins and minerals differ in their roles and importance for health?",
    "Describe the process of digestion and absorption of nutrients in the human body.",
    "What role does fibre play in digestion? Name five fibre containing foods.",
    "Explain the concept of energy balance and its importance in weight management."
]

# Manually created question list
manual_questions = [
    "How often should infants be breastfed?",
    "What are symptoms of pellagra?",
    "How does saliva help with digestion?",
    "water soluble vitamins"
]

query_list = gpt4_questions + manual_questions

In [ ]:
import random
query = random.choice(query_list)

print(f"Query: {query}")

# Get just the scores and indices of top related results
scores, indices = retrieve_relevant_resources(query=query,
                                            embeddings=embeddings)
scores, indices

In [ ]:
def prompt_formatter(query: str, 
                    context_items: list[dict]) -> str:
    # Join context items into one dotted paragraph
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])

    # Create a base prompt with examples to help the model
    # Note: this is very customizable, I've chosen to use 3 examples of the answer style we'd like.
    # We could also write this in a txt file and import it in if we wanted.
    base_prompt = """Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Make sure your answers are as explanatory as possible.
Use the following examples as reference for the ideal answer style.
\nExample 1:
Query: What are the fat-soluble vitamins?
Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, Vitamin E, and Vitamin K. These vitamins are absorbed along with fats in the diet and can be stored in the body's fatty tissue and liver for later use. Vitamin A is important for vision, immune function, and skin health. Vitamin D plays a critical role in calcium absorption and bone health. Vitamin E acts as an antioxidant, protecting cells from damage. Vitamin K is essential for blood clotting and bone metabolism.
\nExample 2:
Query: What are the causes of type 2 diabetes?
Answer: Type 2 diabetes is often associated with overnutrition, particularly the overconsumption of calories leading to obesity. Factors include a diet high in refined sugars and saturated fats, which can lead to insulin resistance, a condition where the body's cells do not respond effectively to insulin. Over time, the pancreas cannot produce enough insulin to manage blood sugar levels, resulting in type 2 diabetes. Additionally, excessive caloric intake without sufficient physical activity exacerbates the risk by promoting weight gain and fat accumulation, particularly around the abdomen, further contributing to insulin resistance.
\nExample 3:
Query: What is the importance of hydration for physical performance?
Answer: Hydration is crucial for physical performance because water plays key roles in maintaining blood volume, regulating body temperature, and ensuring the transport of nutrients and oxygen to cells. Adequate hydration is essential for optimal muscle function, endurance, and recovery. Dehydration can lead to decreased performance, fatigue, and increased risk of heat-related illnesses, such as heat stroke. Drinking sufficient water before, during, and after exercise helps ensure peak physical performance and recovery.
\nNow use the following context items to answer the user query:
{context}
\nRelevant passages: <extract relevant passages from the context here>
User query: {query}
Answer:"""

    # Update base prompt with context items and query   
    base_prompt = base_prompt.format(context=context, query=query)

    # Create prompt template for instruction-tuned model
    dialogue_template = [
        {"role": "user",
        "content": base_prompt}
    ]

    # Apply the chat template
    prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                        tokenize=False,
                                        add_generation_prompt=True)
    return prompt

In [ ]:
query = random.choice(query_list)
print(f"Query: {query}")

# Get relevant resources
scores, indices = retrieve_relevant_resources(query=query,
                                            embeddings=embeddings)
    
# Create a list of context items
context_items = [pages_and_chunks[i] for i in indices]

# Format prompt with context items
prompt = prompt_formatter(query=query,
                        context_items=context_items)
print(prompt)

In [ ]:
%%time

input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = llm_model.generate(**input_ids, max_new_tokens = 512,do_sample = True)
# Decode the output tokens to get the generated text
outputs_text = tokenizer.decode(output[0])
print(f"Query : {query}\n")
print(f"RAG Answer: {outputs_text.replace(prompt," ")}\n")

In [30]:
def ask(query: str, 
        max_new_token: int=512, 
        format_answer_text= True, 
        return_answer_only = True):
    # Get jut the score and indices of top related results
    scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings)

    # RETRIEVAL
    # Create a list of context items
    context_items = [pages_and_chunks[i] for i in indices]

    # Add score to context item
    for i, item in enumerate(context_items):
        item["scores"] = scores[i].cpu()

    # AUGMENTATION
    # Create the prompt and format it with context items

    prompt = prompt_formatter(query=query, context_items=context_items)

    # GENERATION
    # Tokenize the prompt
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)
    # Generate an output of tokens
    output = llm_model.generate(**input_ids, max_new_tokens = max_new_token, do_sample = True)
    
    # Turn the output tokens into text
    output_text = tokenizer.decode(output[0])

    if format_answer_text:
    # Replace special tokens and unnecessary help message
        output_text = output_text.replace(prompt, "").replace("<bos>", "").replace("<eos>", "").replace("Sure, here is the answer to the user query:\n\n", "")

    # Only return the answer without the context items
    if return_answer_only:
        return output_text
    
    return output_text, context_items

In [38]:
query = "Name foods with high protein content"
print(f"Query: {query}")

# Answer query with context and return context 
answer, context_items = ask(query=query, 
                            max_new_token=512,
                            return_answer_only=False)

print(f"Answer:\n")
print_wrapped(answer)
print(f"Context items:")
context_items

Query: Name foods with high protein content
Time taken for dot product similarity search: 0.0003 seconds
Answer:

Sure, here is the information you have requested from the context:  **High-
protein foods:**  * Animal-based foods:     * Lean meats: Round steaks, top
sirloin, extra lean ground beef, pork loin, skinless chicken     * Protein: Soy
beans, nuts, whole milk * Plant-based foods:     * Beans: Kidney beans, lentil,
sunflower seeds  This information should help answer your query.
Context items:


[{'page_number': 411,
  'sentence_chunk': 'Dietary Sources of Protein The protein food group consists of foods made from meat, seafood, poultry, eggs, soy, dry beans, peas, and seeds. According to the Harvard School of Public Health, “animal protein and vegetable protein probably have the same effects on health. It’s the protein package that’s likely to make a difference. ”1 1. \xa0Protein: The Bottom Line. Harvard School of Public Proteins, Diet, and Personal Choices | 411',
  'chunk_char_count': 432,
  'chunk_word_count': 70,
  'chunk_token_count': 108.0,
  'embedding': array([ 3.57394032e-02,  4.69983928e-02,  2.42607947e-03, -1.34758363e-02,
          4.41606231e-02,  1.58367469e-03, -5.75249530e-02,  7.43902400e-02,
         -2.55552586e-02, -5.65149449e-02, -2.50401087e-02,  1.29941420e-03,
          5.23345098e-02,  2.63163447e-02,  1.98242310e-02, -4.84029343e-03,
          1.22621628e-02,  6.03821464e-02,  2.70720292e-02,  2.33721323e-02,
         -3.14021148e-02, -5.45082195e